In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    EvalPrediction,
    DataCollatorForTokenClassification,
)
from Bio.SeqIO.FastaIO import SimpleFastaParser
from peft import LoraConfig, get_peft_model

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [92]:
model = AutoModelForTokenClassification.from_pretrained(
    "Synthyra/ESMplusplus_small", trust_remote_code=True, num_labels=2
).to(device)
tokenizer = model.tokenizer

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Identifying LRR domains

Let's use the training dataset of LRR predictor to identify LRR motifs.
We will be doing a binary classification in this case - either motif or non-motif.
The training dataset is good, but I've noticed that some sequences (e.g. 5YUDA, 4KXFB and 5IRNA) appear to be misannotated, containing no 5IRNAs.
Some sequences are also shorter/longer than their motif annotation - strange.

I need to generate embeddings for each sequence, and a corresponding tensor embedding of binary classifications.
For now, I'll just discard any sequence that does not have any motifs.

In [4]:
sequences = []
labels = []

with open("data/lrrpredictor_training.csv") as f:
    for line in f:
        _, data_type, value = line.strip().split(",")
        if data_type == "sequence":
            sequences.append(value)
        elif data_type == "LRR motifs":
            label = np.zeros(len(value), dtype=int)
            for i, c in enumerate(value):
                if c == "-":
                    label[i] = 0
                else:
                    label[i] = 1
            labels.append(label)

train_sequences, test_sequences, train_labels, test_labels = train_test_split(
    sequences, labels, test_size=0.25, shuffle=True
)

train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset = train_dataset.add_column("labels", train_labels)
test_dataset = test_dataset.add_column("labels", test_labels)

In [93]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [94]:
def compute_metrics_classification(p: EvalPrediction) -> dict[str, float]:
    predictions, labels = p.predictions, p.label_ids
    predictions = predictions[0] if isinstance(predictions, tuple) else predictions
    predictions = np.argmax(predictions, axis=-1)

    mask = labels != -100

    accuracy = (predictions[mask] == labels[mask]).mean()
    return {"accuracy": accuracy}

In [95]:
target_modules = ["layernorm_qkv.1", "out_proj", "query", "key", "value", "dense"]

lora_config = LoraConfig(
    r=8,  # choose lora parameters to your liking
    lora_alpha=16,
    lora_dropout=0.01,
    bias="none",
    target_modules=target_modules,
    modules_to_save=["classifier.0", "classifier.2", "classifier.3"],
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

for param in model.classifier.parameters():
    param.requires_grad = True

In [96]:
from tempfile import gettempdir

tmpdir = gettempdir()

training_args = TrainingArguments(
    output_dir=f"{tmpdir}/LRR_ESMplusplus_small",
    num_train_epochs=10,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir=f"{tmpdir}/LRR_ESMplusplus_small",
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="best",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    label_names=["labels"],
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_classification,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.720500,0.710453,0.596368
2,0.597100,0.576347,0.693142
3,0.461900,0.406112,0.818650
4,0.320900,0.260743,0.918677
5,0.193100,0.171262,0.957351
6,0.129700,0.115006,0.968917
7,0.082700,0.079675,0.978856
8,0.056100,0.062176,0.983013
9,0.047300,0.053231,0.984097
10,0.037300,0.047745,0.984549


/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.wa

TrainOutput(global_step=90, training_loss=0.2646609902381897, metrics={'train_runtime': 82.4111, 'train_samples_per_second': 16.139, 'train_steps_per_second': 1.092, 'total_flos': 2330572282681728.0, 'train_loss': 0.2646609902381897, 'epoch': 10.0})

In [97]:
trainer.save_model("LRR_ESMplusplus_small")

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "LRR_ESMplusplus_small", trust_remote_code=True, num_labels=2
).to(device)
# model = PeftModel.from_pretrained(model, "LRR_ESMplusplus_small").to(device)
tokenizer = model.tokenizer

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [103]:
model.eval()
with open("../../refplantnlr.faa") as handle:
    for id, seq in SimpleFastaParser(handle):
        inputs = tokenizer(seq, return_tensors="pt", padding=True).to(device)
        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1).squeeze()
        pred_str = "".join([str(x) for x in predictions.tolist()])
        print(id)
        print(seq)
        print(pred_str)
        print()

3gG2
MALELVGGALLSAFLQVAFEKLASPQVLDFFRGRKLDEKLLNNLEIKLNSIQALADDAELKQFRDPPVRNWLLKVKDALFDAEDLLDEIQHEISKCQVEAEAEAESQTCTCKVPNFFKSSPVGSFNKEIKSRMEQVLEDLENLASQSGYLGLQNASGVGSGFGGAVSLHSESTSLVVESVIYGRDDDKEMIFNWLTSDIDNCNKLSILSIVGMGGLGKTTLAQHVFNDPRIENKFDIKAWVCVSDEFDVFNVTRTILEAVTKSTDDSRNRETVQGRLREKLTGNKFFLVLDDVWNRNQKEWKDLQTPLNYGASGSKIVVTTRDKKVASIVGSNKTHCLELLQDDHCWRLFTKHAFRDDSHQPNPDFKEIGTKIVEKCKGLPLALTTIGSLLHQKSSISEWEGILKSEIWEFSEEDSSIVPALALSYHHLPSHLKRCFAYCALFPKDYRFDKEGLIQLWMAENFLQCHQQSRSPEKVGEQYFNDLLSRSLFQQSSTVERTPFVMHDLLNDLAKYVCGDICFRLENDQATNIPKTTRHFSVASDHVTCFDGFRTLYNAERLRTFMSLSEEMSFRNYNPWYCKMSTRELFSKFKFLRVLSLSGYYNLTKVPNSVGNLKYLSSLDLSHTEIVKLPESICSLYNLQILKLNGCEHLKELPSNLHKLTDLHRLELIDTEVRKVPAHLGKLKYLQVLMSSFNVGKSREFSIQQLGELNLHGSLSIRQLQNVENPSDALAVDLKNKTHLVELELEWDSDWNPDDSTKERDVIENLQPSKHLEKLTMSNYGGKQFPRWLFNNSLLRVVSLTLKNCKGFLCLPPLGRLPSLKELSIEGLDGIVSINADFFGSSSCSFTSLESLEFSDMKEWEEWECKGVTGAFPRLQRLSIMRCPKLKGHLPEQLCHLNYLKISGWDSLTTIPLDIFPILKELQIWECPNLQRISQGQALNHLETLSMRECPQLESLPEGMHVLLPSLDSLWIDDCPKVEMFPEGGLPSNLKSMG

KeyboardInterrupt: 